# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [1]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 


# Dates
start_date = "11-01-2021"
end_date = "06-13-2025"
date_range = start_date + "--" + end_date
# update_date = "06-18-2025"

# Make sure you have the correct paths

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/"
downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = home + "references/"
originals = downloads + "Andersen/"
saved = originals + "saved/"
temp_files = originals + "temp/"
complete_files = downloads + "complete/"

os.chdir(downloads)

## Read Metadata 

In [2]:
# Read metadata

# os.chdir(saved)
# metadata_normalized = pd.read_csv("metadata_normalized.tsv", delimiter="\t") # Collection dates

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

# metadata = metadata.merge(metadata_normalized, how="outer")
print(metadata.columns)

metadata["name_state"] = metadata["geo_loc_name"].apply(lambda x: x.split("/")[1] if len(x.split("/")) > 1 else x.split("/")[0])

# Find only >= last date using Release Date from metadata 
metadata["ReleaseDate"] = metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
# print(metadata["ReleaseDate"])
metadata = metadata[metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")]
# Find only <= update date using Release Date from metadata
metadata = metadata[metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")]

# Get rid of certain runs
os.chdir(saved)
runs_to_remove = pd.read_csv("andersen-lab-seqs-to-filter.csv")
print(runs_to_remove)
for run in runs_to_remove["Run"].values:
    metadata = metadata[metadata["Run"] != run]

print(len(metadata)) # 6053 rows between 1/1/2024 and 4/14/2025
display(metadata)
print(metadata["Library Name"])

9808
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
              Run
0     SRR24839736
1     SRR24839063
2     SRR24839065
3     SRR24839713
4     SRR24839570
...           ...
1310  SRR24843048
1311  SRR24839118
1312  SRR24839711
1313  SRR24843405
1314  SRR24843040

[1315 rows x 1 columns]
8444


,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,create_date,version,Sample Name,SRA Study,serotype,isolation_source,BioSample Accession,is_retracted,retraction_detection_date_utc,name_state
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,2024-04-20T18:12:00Z,1,24-008354-001-original,SRP503016,H5N1,NaN,SRS21079812,False,NaN,USA
1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,2024-04-20T18:12:00Z,1,24-009108-005-original,SRP503016,H5N1,NaN,SRS21079811,False,NaN,USA
2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,2024-04-20T18:12:00Z,1,24-009108-004-original,SRP503016,H5N1,NaN,SRS21079810,False,NaN,USA
3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,2024-04-20T18:12:00Z,1,24-009108-003-original,SRP503016,H5N1,NaN,SRS21079809,False,NaN,USA
4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,2024-04-20T18:12:00Z,1,24-009108-002-original,SRP503016,H5N1,NaN,SRS21079808,False,NaN,USA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9754,SRR33943358,WGS,148.58,145468815,PRJNA1102327,SAMN49008963,Viral,50309372,USDA-NVSL,2025,...,2025-06-11 17:51:42,1,25-013594-001,SRP503016,NaN,milk,SRS25347775,False,NaN,USA
9755,SRR33943359,WGS,148.65,141682464,PRJNA1102327,SAMN49008954,Viral,52273363,USDA-NVSL,2025,...,2025-06-11 17:51:39,1,25-016317-001,SRP503016,NaN,milk,SRS25347774,False,NaN,USA
9756,SRR33943360,WGS,148.56,109583296,PRJNA1102327,SAMN49008953,Viral,40873907,USDA-NVSL,2025,...,2025-06-11 17:51:39,1,25-015080-001,SRP503016,NaN,"MILK, BULK TANK",SRS25347773,False,NaN,USA
9757,SRR33943301,WGS,148.82,146047742,PRJNA980729,SAMN49008919,Viral,51264801,USDA-NVSL,2025,...,2025-06-11 17:50:24,1,25-016837-002,SRP441379,NaN,OROPHARYNGEAL SWAB,SRS25347770,False,NaN,USA


0               24-008354-001-original
1               24-009108-005-original
2               24-009108-004-original
3               24-009108-003-original
4               24-009108-002-original
                     ...              
9754    25-013594-001-original-repeat2
9755            25-016317-001-original
9756            25-015080-001-original
9757            25-016837-002-original
9758            25-016837-001-original
Name: Library Name, Length: 8444, dtype: object


In [3]:
# Get list of genotypes

os.chdir(home + "references/")

genotypes_df = pd.read_excel("genotype_key.xlsx")

genotypes = list(genotypes_df["Genotype"])

print(genotypes)

# genotypes = ["B3.13", "D1.1"]

# genotypes = ["B3.2"] #, "B3.2", "B3.6", "B3.7", "B3.5", "A3"]

['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'B1.1', 'B1.2', 'B1.3', 'B2.1', 'B2.2', 'B3.1', 'B3.2', 'B3.3', 'B3.4', 'B3.5', 'B3.6', 'B4.1', 'B5.1', 'Minor01', 'Minor04', 'Minor07', 'Minor08', 'Minor09', 'Minor10', 'Minor11', 'Minor12', 'Minor13', 'Minor14', 'Minor15', 'Minor16', 'Minor17', 'Minor18', 'Minor19', 'Minor24', 'Minor25', 'Minor26', 'Minor27', 'Minor28', 'Minor29', 'Minor30', 'Minor31', 'Minor32', 'Minor33', 'Minor34', 'Minor35', 'Minor36', 'Minor37', 'Minor38', 'Minor39', 'Minor40', 'Minor41', 'Minor42', 'Minor43', 'Minor44', 'Minor45', 'Minor46', 'Minor47', 'Minor48', 'B3.7', 'Minor50', 'Minor51', 'C1.1', 'Minor52', 'Minor53', 'B3.11', 'Minor55', 'Minor56', 'Minor57', 'Minor58', 'B3.10', 'C2.1', 'Minor60', 'Minor61', 'B3.8', 'Minor62', 'Minor63', 'B3.12', 'Minor65', 'Minor66', 'Minor67', 'B3.9', 'Minor70', 'Minor71', 'B3.13', 'Minor73', 'Minor74', 'Minor75', 'Minor76', 'Minor77', 'Minor78', 'Minor79', 'Minor80', 'Minor81', 'Minor82', 'Minor83', 'Minor84', 'C3.1', 'Minor86', 'Mino

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use genoflu_results.tsv

## Get genotype, specific geolocation

In [4]:
# Get genotype from genoflu_results.tsv

os.chdir(metadata_folder)

genoflu_results = pd.read_csv("genoflu_results.tsv", delimiter="\t")

genoflu_results["Run"] = genoflu_results["sample"]

metadata = metadata.merge(genoflu_results, on="Run", how="inner")
print(metadata)
# metadata = metadata[~metadata["Genotype"].str.contains('Not assigned')] # Do not include non-assigned genotypes
metadata = metadata[metadata["Genotype"].isin(genotypes)]

# Get only the genotypes we want: B3.13 and D1.1

# b313_and_d11_only = genoflu_results[(genoflu_results["Genotype"] == "B3.13") | (genoflu_results["Genotype"] == "D1.1")]
# b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
# b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")

# metadata = metadata.merge(b313_and_d11_only, on=["Run", "Genotype"], how="inner")

print(len(metadata)) 

display(metadata)

              Run Assay Type  AvgSpotLen      Bases    BioProject  \
0     SRR28752446        WGS      146.11   93605195  PRJNA1102327   
1     SRR28752447        WGS      241.29   86080323  PRJNA1102327   
2     SRR28752448        WGS      250.30   75035343  PRJNA1102327   
3     SRR28752449        WGS      146.61   59363690  PRJNA1102327   
4     SRR28752450        WGS      251.31  119232569  PRJNA1102327   
...           ...        ...         ...        ...           ...   
8439  SRR33943358        WGS      148.58  145468815  PRJNA1102327   
8440  SRR33943359        WGS      148.65  141682464  PRJNA1102327   
8441  SRR33943360        WGS      148.56  109583296  PRJNA1102327   
8442  SRR33943301        WGS      148.82  146047742   PRJNA980729   
8443  SRR33943302        WGS      148.87  176312039   PRJNA980729   

         BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0     SAMN41019184          Viral  30074178   USDA-NVSL            2024  ...   
1     SAMN4

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,name_state,sample,date,File Name,Genotype,"Genotype List Used, >=98.0%",Genotype Sample Title List,Genotype Percent Match List,Genotype Mismatch List,Genotype Average Depth of Coverage List
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,USA,SRR28752446,2025-05-09_10-47-05,SRR28752446.fa,B3.13,"NS:am1.1, PB1:am4, PB2:am2.2, HA:ea1, MP:ea1, ...","am1.1:22-010085-001:NS, am4:23-001855-001:PB1,...","99.17%, 99.52%, 98.90%, 98.77%, 98.88%, 99.16%...","7, 11, 25, 21, 11, 18, 10, 14",Ran on FASTA - No Coverage Report
1,SRR28752447,WGS,241.29,86080323,PRJNA1102327,SAMN41019237,Viral,28164109,USDA-NVSL,2024,...,USA,SRR28752447,2025-05-09_10-45-55,SRR28752447.fa,B3.13,"NP:am8, MP:ea1, PB2:am2.2, HA:ea1, NA:ea1, PA:...","am8:23-032005-001:NP, ea1:22-003707-003:MP, am...","99.40%, 98.88%, 98.86%, 98.77%, 99.08%, 99.16%...","9, 11, 26, 21, 13, 18, 6, 10",Ran on FASTA - No Coverage Report
2,SRR28752448,WGS,250.30,75035343,PRJNA1102327,SAMN41019236,Viral,24547283,USDA-NVSL,2024,...,USA,SRR28752448,2025-05-09_10-47-06,SRR28752448.fa,B3.13,"HA:ea1, NA:ea1, MP:ea1, PA:ea1, PB1:am4, PB2:a...","ea1:22-003707-003:HA, ea1:22-003707-003:NA, ea...","98.77%, 99.08%, 98.88%, 99.21%, 99.56%, 98.86%...","21, 13, 11, 17, 10, 26, 7, 10",Ran on FASTA - No Coverage Report
3,SRR28752449,WGS,146.61,59363690,PRJNA1102327,SAMN41019235,Viral,19686302,USDA-NVSL,2024,...,USA,SRR28752449,2025-05-09_10-46-33,SRR28752449.fa,B3.13,"MP:ea1, PA:ea1, PB2:am2.2, NP:am8, NS:am1.1, N...","ea1:22-003707-003:MP, ea1:22-003707-003:PA, am...","98.88%, 99.16%, 98.86%, 99.33%, 99.28%, 99.08%...","11, 18, 26, 10, 6, 13, 21, 10",Ran on FASTA - No Coverage Report
4,SRR28752450,WGS,251.31,119232569,PRJNA1102327,SAMN41019234,Viral,38846827,USDA-NVSL,2024,...,USA,SRR28752450,2025-05-09_10-47-05,SRR28752450.fa,B3.13,"PA:ea1, PB2:am2.2, MP:ea1, NS:am1.1, PB1:am4, ...","ea1:22-003707-003:PA, am2.2:22-010445-001:PB2,...","99.21%, 98.82%, 98.88%, 99.17%, 99.56%, 98.77%...","17, 27, 11, 7, 10, 21, 10, 13",Ran on FASTA - No Coverage Report
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8439,SRR33943358,WGS,148.58,145468815,PRJNA1102327,SAMN49008963,Viral,50309372,USDA-NVSL,2025,...,USA,SRR33943358,2025-06-13_07-18-34,SRR33943358.fa,B3.13,"PA:ea1, NA:ea1, PB2:am2.2, NP:am8, HA:ea1, PB1...","ea1:22-003707-003:PA, ea1:22-003707-003:NA, am...","98.84%, 98.65%, 98.51%, 98.86%, 98.12%, 99.56%...","25, 19, 34, 17, 32, 10, 9, 10",Ran on FASTA - No Coverage Report
8440,SRR33943359,WGS,148.65,141682464,PRJNA1102327,SAMN49008954,Viral,52273363,USDA-NVSL,2025,...,USA,SRR33943359,2025-06-13_07-18-34,SRR33943359.fa,B3.13,"HA:ea1, PB1:am4, NA:ea1, MP:ea1, NP:am8, PA:ea...","ea1:22-003707-003:HA, am4:23-001855-001:PB1, e...","98.42%, 99.34%, 98.58%, 98.78%, 98.93%, 98.79%...","27, 15, 20, 12, 16, 26, 33, 8",Ran on FASTA - No Coverage Report
8441,SRR33943360,WGS,148.56,109583296,PRJNA1102327,SAMN49008953,Viral,40873907,USDA-NVSL,2025,...,USA,SRR33943360,2025-06-13_07-18-34,SRR33943360.fa,B3.13,"PB1:am4, MP:ea1, NP:am8, NA:ea1, PB2:am2.2, HA...","am4:23-001855-001:PB1, ea1:22-003707-003:MP, a...","99.12%, 98.98%, 98.93%, 98.72%, 98.51%, 98.30%...","20, 10, 16, 18, 34, 29, 22, 11",Ran on FASTA - No Coverage Report
8442,SRR33943301,WGS,148.82,146047742,PRJNA980729,SAMN49008919,Viral,51264801,USDA-NVSL,2025,...,USA,SRR33943301,2025-06-13_07-18-34,SRR33943301.fa,D1.1,"PB2:am24, NA:am4N1, MP:ea3, NS:ea3, PA:am4, HA...","am24:24-030039-001:PB2, am4N1:24-030039-001:NA...","99.56%, 99.04%, 100.00%, 98.93%, 98.75%, 99.47...","10, 10, 0, 9, 27, 9, 17, 7",Ran on FASTA - No Coverage Report


In [ ]:
# Get specific geolocation and name_state from genbank_mapping.tsv

genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run", keep="first") # Drop duplicates
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2]) # Get the name of the state

# print(genbank_mapping)

# metadata_genbank = metadata.merge(genbank_mapping, on=["Run"]) # Only include data that has states

# Get geolocation for second state attribute

os.chdir(home + "references/")
state_ref = pd.read_csv("states_ref.csv")
metadata["Geo_Location"] = metadata["name_state"].apply(lambda x: 
                                                        state_ref.loc[state_ref["Abbreviation"].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0] 
                                                        + "-" + 
                                                        x
                                                        if state_ref["Abbreviation"].str.contains("|".join((x.replace(", ", " ").split(" "))), regex=True).any()
                                                        else state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Country'].iloc[0]
                                                        + "-" + 
                                                        state_ref.loc[state_ref['State'].str.contains('|'.join(x.replace(', ', ' ').split(' ')), regex=True), 'Abbreviation'].iloc[0] 
                                                        if state_ref["State"].str.contains("|".join((x.replace(", ", " ").split(" "))), regex=True).any() 
                                                        else x)

print(state_ref.loc[state_ref['State'].str.contains('|'.join("Kentucky, whatever".replace(',', ' ').split(' ')), regex=True), 'Abbreviation'])
print(state_ref['State'].str.contains('|'.join("Kentucky, whatever".replace(', ', ' ').split(' ')), regex=True))
print(metadata["name_state"])
print(metadata["Geo_Location"])
print(len(metadata))
display(metadata) # Maybe there is no state information since 3/18/2025?

IndexError: single positional indexer is out-of-bounds

In [ ]:
# # If no states

# metadata_genbank = metadata

# metadata_genbank["name_state"] = "USA"

# metadata_genbank["Geo_Location"] = "USA"

# display(metadata_genbank)

## Collection Dates

If date is N/A, try finding it first

In [ ]:

# # Get all dates
# metadata["Collection_Date_Specific"] = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata) if "-" not in x else x)

# # Save this so we don't have to do it again

# # os.chdir(temp_files)
# metadata.to_csv("metadata_genbank.csv")

In [ ]:
os.chdir(saved)

metadata_to_merge = pd.read_csv("metadata_genbank_" + date_range + ".csv")
metadata = pd.merge(metadata, metadata_to_merge, how="left")
# metadata = pd.read_csv("metadata_genbank.csv")
print(len(metadata))

7959


C:\Users\maksi\AppData\Local\Temp\ipykernel_22472\3732131206.py:3: DtypeWarning: Columns (41,46,48) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_to_merge = pd.read_csv("metadata_genbank_" + date_range + ".csv")


In [ ]:
date = "2025" 

date = dateutil.parser.parse(date, default=datetime(2000, 1, 1))

print(date.month)

1


In [ ]:
# # # Upload saved data -- if doing this, make sure the above cell is commented out
# # os.chdir(temp_files + "saved/")
# # metadata_genbank = pd.read_csv("metadata_genbank.csv")
# # os.chdir(temp_files)

# # Get only updated dates

# # unknown_dates = metadata[(metadata["Collection_Date"] == "2024") | (metadata["Collection_Date"] == "2025")] # Dates we don't have

# def find_known_dates(x, df):
    
#     try:
#         date = metadata[metadata["BioSample"] == x]["Collection_Date"].values[0]
#     # print(date)
#         # print(date)
#         # if len(str(date)) == 4: # If this is just a year
#         #     date = search_collection_date(x, df)
#         # else:
#         date = dateutil.parser.parse(date, default=datetime(2000, 1, 1)) # .strftime("%Y-%m-%d") # If a date already exists
#         if date.day == dateutil.parser.parse("1/1/2000").day and date.month == dateutil.parser.parse("1/1/2000").month:
#             print("year only")
#             date = search_collection_date(x, df)
#         print("Success", date)
#     except:
#         date = search_collection_date(x, df) # If it's not parseable as a date
#     return date # If statement in lambda function will search for the "just year" values
        

# # years = ["2021", "2022", "2023", "2024", "2025"]
# # unknown_dates = metadata[metadata["Collection_Date"].isin(years)]
# # # known_dates = metadata[(metadata["Collection_Date"] != "2024") & (metadata["Collection_Date"] != "2025")] # Dates we've already gotten
# # known_dates = metadata[~metadata["Collection_Date"].isin(years)]

# # Get new dates also 
# # new_dates = metadata["BioSample"].apply(lambda x: search_collection_date(x, metadata_genbank) if )

# updated_dates = metadata["BioSample"].apply(lambda x: find_known_dates(x, metadata)) # Update unknown dates, if possible
# metadata["Collection_Date"] = updated_dates

# # metadata = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # updated_unknown_dates = unknown_dates["BioSample"].apply(lambda x: search_collection_date(x, unknown_dates)) # Update unknown dates, if possible
# # unknown_dates["Collection_Date"] = updated_unknown_dates

# # metadata = pd.concat([known_dates, unknown_dates], ignore_index=True, sort=True)

# # Save this so we don't have to do it again

# # os.chdir(temp_files)
# # metadata.to_csv("metadata_genbank_" + date_range + ".csv")

# # display(metadata)

In [ ]:
# os.chdir(temp_files)
# metadata.to_csv("metadata_genbank_" + date_range + ".csv")

# If no collection dates

# metadata_genbank["Collection_Date_Specific"] = metadata_genbank["Collection_Date"]

## Get host type

In [ ]:
# create a mask, where is True if the host does not exist
print(metadata["Host"])

mask = metadata["Host"].isna()

# choose between the original value and split isolate using the mask
metadata["Host"] = np.where(mask, metadata["isolate"].apply(lambda x: x if x != x or "/" not in x or len(x.split("/")) < 2 else x.split("/")[1]), metadata["Host"]) #  if "/" in metadata["isolate"] else metadata["Host"])
metadata["Host"] = metadata["Host"].apply(lambda x: x.lower() if x == x else x)

# print(metadata["Host"])

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

# print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

os.chdir(home + "references/")

animals_ref = pd.read_csv("animals_ref.csv") # Upload animals ref

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort2.csv") # Make sure name is different to avoid overwriting the first reference 

print(metadata["Host"])
# print(metadata["isolate"])

0       Blackbird
1          Cattle
2          Cattle
3          Cattle
4          Cattle
          ...    
7954       CATTLE
7955       CATTLE
7956       CATTLE
7957      CHICKEN
7958      CHICKEN
Name: Host, Length: 7959, dtype: object
[]
                  avian               cattle        feline   other_mammal  \
0      great_horned_owl            dairy_cow           cat     deer mouse   
1          common_raven               cattle  domestic_cat    house_mouse   
2         cooper's_hawk  cattle milk product     feral_cat          skunk   
3          coopers_hawk          bovine_milk        feline  striped_skunk   
4               peafowl              bovine   domestic-cat     norway rat   
..                  ...                  ...           ...            ...   
466         anser anser                  NaN           NaN            NaN   
467    northern harrier                  NaN           NaN            NaN   
468              mergus                  NaN           NaN        

In [ ]:
# Get animals from animal reference
os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata, animals_ref) # Get host type

metadata["years"] = metadata["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

## Make names using all the attributes we collected

In [ ]:
for num, collection_date in enumerate(metadata["Collection_Date"]):
    if collection_date != collection_date: # if nan
         collection_date = "missing"
    try:
        print(collection_date)
        if collection_date.month == datetime.parser.parse("1/1/2000").month and collection_date.day == datetime.parser.parse("1/1/2000").day:
            # If not a valid collection date, but has year
            
            year = collection_date.year
            metadata.loc[num, "Collection_Date"] = year
            # if collection_date != collection_date: # If nan
        #     metadata.loc[num, "Collection_Date"] = metadata.loc[num, "years"]
        else: # If actual date
                
            # if len(str(collection_date)) == 4: # If it's a year
                # print("caught")
                metadata.loc[num, "Collection_Date"] = collection_date
    except:
         metadata.loc[num, "Collection_Date"] = collection_date
        # else:
        #     try:
        #         parsed_date = dateutil.parser.parse(collection_date)
        #         date = parsed_date.strftime("%Y-%m-%d") # Make sure it doesn't default to today, if just a year
        #         metadata.loc[num, "Collection_Date"] = date
        #     except: # If no date at all
        #         metadata.loc[num, "Collection_Date"] = collection_date

    # metadata = metadata.dropna(thresh=2)



2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024
2024


In [ ]:
# Make names

metadata = metadata.fillna("")

# + metadata["BioSample"] + "|" 
names = ">" + metadata["Run"] + "|" + "A/" + metadata["Host"] + "/" + metadata["name_state"] + "/" + metadata["Sample Name"] + "/" + metadata["years"].apply(lambda x: str(x)) + "|" + metadata["serotype"] + "|" + metadata["Geo_Location"] + "|" + metadata["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata["Host_Type"] + "|" + metadata["Genotype"]

metadata["Name"] = names

# metadata_genbank.to_csv("metadata_genbank_named.csv")

display(metadata["Name"])

0       >SRR28752446|A/blackbird/USA/24-008354-001-ori...
1       >SRR28752447|A/cattle/USA/24-009108-005-origin...
2       >SRR28752448|A/cattle/USA/24-009108-004-origin...
3       >SRR28752449|A/cattle/USA/24-009108-003-origin...
4       >SRR28752450|A/cattle/USA/24-009108-002-origin...
                              ...                        
7954    >SRR33943358|A/cattle/USA/25-013594-001/2025||...
7955    >SRR33943359|A/cattle/USA/25-016317-001/2025||...
7956    >SRR33943360|A/cattle/USA/25-015080-001/2025||...
7957    >SRR33943301|A/chicken/USA/25-016837-002/2025|...
7958    >SRR33943302|A/chicken/USA/25-016837-001/2025|...
Name: Name, Length: 7959, dtype: object

In [ ]:
# Drop duplicate runs 
metadata = metadata.drop_duplicates(subset="Run", keep="first")

In [ ]:
print(metadata)
metadata.to_csv("metadata_test.csv")

              Run Assay Type AvgSpotLen      Bases    BioProject  \
0     SRR28752446        WGS     146.11   93605195  PRJNA1102327   
1     SRR28752447        WGS     241.29   86080323  PRJNA1102327   
2     SRR28752448        WGS      250.3   75035343  PRJNA1102327   
3     SRR28752449        WGS     146.61   59363690  PRJNA1102327   
4     SRR28752450        WGS     251.31  119232569  PRJNA1102327   
...           ...        ...        ...        ...           ...   
7954  SRR33943358        WGS     148.58  145468815  PRJNA1102327   
7955  SRR33943359        WGS     148.65  141682464  PRJNA1102327   
7956  SRR33943360        WGS     148.56  109583296  PRJNA1102327   
7957  SRR33943301        WGS     148.82  146047742   PRJNA980729   
7958  SRR33943302        WGS     148.87  176312039   PRJNA980729   

         BioSample BioSampleModel     Bytes Center Name Collection_Date  ...  \
0     SAMN41019184          Viral  30074178   USDA-NVSL            2024  ...   
1     SAMN41019237     

## Make FASTA files

In [ ]:
# Get information to create the fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]
pairs = []
fasta_files = {}

for genotype in genotypes: # ["B3.13", "D1.1"]:
    for segment in segments:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata[metadata["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata[metadata["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()
        break 

In [ ]:
# print(fasta_files.keys())

In [ ]:
# Create fasta files 

# os.chdir(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
os.chdir(originals + "complete/")
names = []
for pair in fasta_files.keys():
    # output_path = complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/" + pair + "_andersen_updated_" + update_date + ".fasta" 
    output_path = originals + "complete/" + pair + "_" + date_range + "_andersen_updated.fasta"

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        names.append(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

print(len(names)/8)

>SRR29740415|A/red-tailed hawk/USA: Kittery, ME/24-007611-001/2023||USA: Kittery, ME|2023-12-29|avian|A2
>SRR29740446|A/sanderling/USA: Dennis, MA/23-015884-008/2023||USA: Dennis, MA|2023-02-21|avian|A2
>SRR29740465|A/canada goose/USA: Wellfleet, MA/24-004865-021/2024||USA: Wellfleet, MA|2024-01-23|avian|A2
>SRR29740479|A/red-tailed hawk/USA: Centerville, MA/23-015884-005/2023||USA: Centerville, MA|2023-02-11|avian|A2
>SRR29740484|A/canada goose/USA: South Boston, MA/24-004865-002/2024||USA-SC|2024-01-31|avian|A2
>SRR29740495|A/turkey vulture/USA: Tiverton, RI/24-014130-005/2024||USA: Tiverton, RI|2024-04-19|avian|A2
>SRR29740497|A/turkey vulture/USA: Tiverton, RI/24-014130-004/2024||USA: Tiverton, RI|2024-04-19|avian|A2
>SRR32499025|A/wood duck/USA/24-008552-015/2024||USA|2024|avian|A2
>SRR29740415|A/red-tailed hawk/USA: Kittery, ME/24-007611-001/2023||USA: Kittery, ME|2023-12-29|avian|A2
>SRR29740446|A/sanderling/USA: Dennis, MA/23-015884-008/2023||USA: Dennis, MA|2023-02-21|avian|A2

## De-Duplication

In [ ]:
# De-duplication 

# gisaid = downloads + "GISAID/complete/all_genotypes/11-01-2021--06-13-2025_all_genotypes_Antarctica_North_America_South_America/"
    # gisaid = downloads + "Cats/Datasets/GISAID/"

# for genotype in genotypes:

    # gisaid = downloads + "GISAID/complete/" + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "_Antarctica_North_America_South_America/"
    
gisaid = downloads + "GISAID/complete/" + date_range + "_all_genotypes_Antarctica_North_America_South_America/"

os.chdir(gisaid)

# Gisaid 
dfs_gisaid_list = []
dfs_gisaid = create_dataframes(gisaid)
dfs_gisaid_list.append(dfs_gisaid)

# dfs_gisaid = {}
# for df_gisaid in dfs_gisaid_list:
#     dfs_gisaid = dfs_gisaid | df_gisaid
# # dfs_gisaid2 = create_dataframes(gisaid2)

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [ ]:
# Do the same with Andersen 

# dfs_andersen = create_dataframes(complete_files + "/B3_13_D1_1/" + date_range + "_B3_13_D1_1/")
dfs_andersen = create_dataframes(originals + "complete/")

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [ ]:
# os.chdir(downloads)
# dfs_gisaid["B3.13_HA"].to_csv

In [ ]:
for key in dfs_andersen.keys():
    dataframes = dfs_andersen[key]
    print(key)

print(dfs_andersen)

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA


In [ ]:
print(len(list(dfs_gisaid.keys())))
print(len(list(dfs_andersen.keys())))

632
968


In [ ]:
# Merge dataframes and drop duplicates

full_dfs = defaultdict(list)
# same = []
# andersen = set()
# gisaid = set()

for i, andersen_key in enumerate(dfs_andersen.keys()):
    if len(dfs_andersen[andersen_key]) > 0:
        for j, gisaid_key in enumerate(dfs_gisaid.keys()):
            if andersen_key == gisaid_key:
                # same.append(gisaid_key)
        # gisaid_key = list(dfs_gisaid.keys())[i]
        # gisaid2_key = list(dfs_gisaid2.keys())[i]

                andersen_df = dfs_andersen[andersen_key][0]
                print(len(andersen_df))
                # print(andersen_df)
                gisaid_df = dfs_gisaid[gisaid_key][0]
                print(len(gisaid_df))
                # gisaid2_df = dfs_gisaid2[gisaid2_key][0]

                # print(pd.concat([gisaid_df, andersen_df]).drop_duplicates())

                full_df = pd.concat([andersen_df, gisaid_df], ignore_index=True)
                print("len full df:", len(full_df))
                # test = len(full_df.drop_duplicates(subset="isolate_partial"))

                dedup_df = full_df.drop_duplicates(subset="isolate_partial", keep="last")

                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print((full_df.loc[full_df.duplicated(subset="isolate_partial")]))
                # print(full_df)
                
                # print("Keeping nothing: ", test)
                
                print("len deduplicated:", len(dedup_df))
                full_dfs[andersen_key].append(dedup_df)
            # else:
                # gisaid.add(gisaid_key)
                # andersen.add(andersen_key)
    
    # break 


# print(full_dfs)
# print(len(full_dfs))
# print(319*8)
# print(len(same))
# print(len(andersen))
# print(len(gisaid))

0
729
len full df: 729
len deduplicated: 601
0
729
len full df: 729
len deduplicated: 601
0
729
len full df: 729
len deduplicated: 601
0
729
len full df: 729
len deduplicated: 601
0
729
len full df: 729
len deduplicated: 601
0
729
len full df: 729
len deduplicated: 601
0
729
len full df: 729
len deduplicated: 601
0
729
len full df: 729
len deduplicated: 601
8
497
len full df: 505
len deduplicated: 491
8
497
len full df: 505
len deduplicated: 491
8
497
len full df: 505
len deduplicated: 491
8
497
len full df: 505
len deduplicated: 491
8
497
len full df: 505
len deduplicated: 491
8
497
len full df: 505
len deduplicated: 491
8
497
len full df: 505
len deduplicated: 491
8
497
len full df: 505
len deduplicated: 491
96
318
len full df: 414
len deduplicated: 325
96
318
len full df: 414
len deduplicated: 325
96
318
len full df: 414
len deduplicated: 325
96
318
len full df: 414
len deduplicated: 325
96
318
len full df: 414
len deduplicated: 325
96
318
len full df: 414
len deduplicated: 325
96
3

In [ ]:
# # If none in one database, only use the other and drop duplicates

# full_dfs = defaultdict(list)
# for key in dfs_andersen.keys():
#     print(key)
# # for key in ["D1.3"]:
#     dataframes = dfs_andersen[key]
#     for i, df in enumerate(dataframes):
#         print(i)
#         try:
#             full_df = df.merge(dfs_gisaid[key][i], how="outer")
#             # print(full_df)
#             full_df = full_df.drop_duplicates(subset=["isolate_partial"])
#             full_dfs[key].append(full_df)
#         except:
#             print("Failed to merge dataframes in ", key)
#             full_dfs[key].append(dataframes[i])

## Create FASTA files combining Andersen and GISAID

In [ ]:
# Create FASTA files per segment

combined_files = downloads + "Combinations/GISAID_Andersen/" # B3_13_D1_1/" + date_range + "_B3_13_D1_1/"

os.chdir(combined_files)
for pair in full_dfs.keys():
    print(pair)
    output_path = combined_files + pair + "_combined_" + date_range + ".fasta" 

    output_file = open(output_path, "w")
    for item in full_dfs[pair]:
        # for item in item:
        # item = fasta_files[pair]
        for index, row in item.iterrows():
            name = item.loc[index, "full_header"]
            sequence = item.loc[index, "sequence"]
            # print(name)
        # First is header, second is sequence
        # print(value)
            output_file.write(name)
            output_file.write(sequence)
    output_file.close()

A1_HA
A1_MP
A1_NA
A1_NP
A1_NS
A1_PA
A1_PB1
A1_PB2
A2_HA
A2_MP
A2_NA
A2_NP
A2_NS
A2_PA
A2_PB1
A2_PB2
A3_HA
A3_MP
A3_NA
A3_NP
A3_NS
A3_PA
A3_PB1
A3_PB2
A4_HA
A4_MP
A4_NA
A4_NP
A4_NS
A4_PA
A4_PB1
A4_PB2
A5_HA
A5_MP
A5_NA
A5_NP
A5_NS
A5_PA
A5_PB1
A5_PB2
A6_HA
A6_MP
A6_NA
A6_NP
A6_NS
A6_PA
A6_PB1
A6_PB2
B1.1_HA
B1.1_MP
B1.1_NA
B1.1_NP
B1.1_NS
B1.1_PA
B1.1_PB1
B1.1_PB2
B1.2_HA
B1.2_MP
B1.2_NA
B1.2_NP
B1.2_NS
B1.2_PA
B1.2_PB1
B1.2_PB2
B1.3_HA
B1.3_MP
B1.3_NA
B1.3_NP
B1.3_NS
B1.3_PA
B1.3_PB1
B1.3_PB2
B2.1_HA
B2.1_MP
B2.1_NA
B2.1_NP
B2.1_NS
B2.1_PA
B2.1_PB1
B2.1_PB2
B2.2_HA
B2.2_MP
B2.2_NA
B2.2_NP
B2.2_NS
B2.2_PA
B2.2_PB1
B2.2_PB2
B3.10_HA
B3.10_MP
B3.10_NA
B3.10_NP
B3.10_NS
B3.10_PA
B3.10_PB1
B3.10_PB2
B3.11_HA
B3.11_MP
B3.11_NA
B3.11_NP
B3.11_NS
B3.11_PA
B3.11_PB1
B3.11_PB2
B3.12_HA
B3.12_MP
B3.12_NA
B3.12_NP
B3.12_NS
B3.12_PA
B3.12_PB1
B3.12_PB2
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
B3.1_HA
B3.1_MP
B3.1_NA
B3.1_NP
B3.1_NS
B3.1_PA
B3.1_PB1
B3.1_PB2
B3.2_HA
